# Week 7: Hashing — Intuition & Indexing — PHASE 3 "Choosing Between O(n), O(log n), O(1)"

*📚 Data Structures & Algorithms · ⏱️ 3 Hours · 👨‍🏫 Dr. Arif Solmaz*

## 🎯 Learning Objectives

1. Understand what hashing is and why it matters for fast data access
2. Explain how Python's `dict` achieves O(1) average-case lookup
3. Describe the concept of a hash function: input → number → index
4. Identify what collisions are and common strategies to handle them
5. Build a simple hash table from scratch using a Python list
6. Benchmark `dict` lookup vs `list` search and interpret the results

## 🎯 Core Mastery Connection

O(1) average lookup — the fastest possible. But at what cost? Collisions, memory, and the requirement that keys be hashable. This week you see the ultimate data structure trade-off: hash tables give you constant-time access but use more memory and can degrade with collisions. You will build a hash table from scratch and benchmark it against list search to prove the O(1) vs O(n) difference with real measurements.

---
## 🧭 Three-Hour Interactive Studio Plan

**Audience:** Mechatronics Engineering students  
**Weekly focus:** Week 7: Hashing — Intuition & Indexing — PHASE 3 "Choosing Between O(n), O(log n), O(1)"

**Professional lens:** indexing device IDs and caching repeated sensor lookups.

| Time | Learning cycle |
|---|---|
| 00:00–00:10 | Launch question, prior-knowledge retrieval, outcomes |
| 00:10–00:50 | Concept cycle 1: explain → predict → test |
| 00:50–01:00 | Checkpoint 1, student questions, peer explanation |
| 01:00–01:10 | Break |
| 01:10–01:50 | Concept cycle 2: worked example → variation → discussion |
| 01:50–02:00 | Checkpoint 2 and misconception repair |
| 02:00–02:10 | Break |
| 02:10–02:40 | Core in-class practice with instructor circulation |
| 02:40–02:50 | Checkpoint 3: exam bridge and professional transfer |
| 02:50–03:00 | Open questions, summary, and exit ticket |

The official start and finish times are followed as published in the timetable. Ask questions at any point; the scheduled checkpoints guarantee additional question time. Checkpoints are private self-checks in this runtime—no identity, upload, homework, or instructor dashboard.


In [ ]:
# Run once. This pulse stays only in the current Colab runtime.
_studio_pulses = {}

def studio_pulse(number, response, minimum_words=8):
    words = str(response).strip().split()
    ready = len(words) >= minimum_words
    _studio_pulses[int(number)] = ready
    if ready:
        print(f"✅ Checkpoint {number}: explanation recorded locally ({len(words)} words).")
    else:
        print(f"🟡 Checkpoint {number}: explain your reasoning in at least {minimum_words} words, then retry.")
    print("Nothing is transmitted or stored for grading.")
    return ready

print("✅ Local studio checkpoints ready")

---
## 📦 Setup

Run this cell first to load the required packages.

In [ ]:
import matplotlib.pyplot as plt

import random
import time

---
## Part 1: What Is Hashing?

### The "Magic Function" Analogy

Imagine you have a **huge library** with thousands of books. You could:

| Approach | How it works | Speed |
|----------|-------------|-------|
| **Linear search** | Walk shelf by shelf until you find your book | Slow — O(n) |
| **Sorted + binary search** | Books are alphabetical; flip to the middle, narrow down | Better — O(log n) |
| **Hashing (magic function)** | A formula tells you *exactly* which shelf & slot | Instant — O(1) avg |

**Hashing** is the idea of converting a *key* (like a book title) into a *number* that tells you **where to store or find** the item — no searching required!

**Figure 1.1** — Python's built-in `hash()` function in action

In [ ]:
# Python has a built-in hash() function
# It converts objects into integers

print("hash('apple')  =", hash('apple'))
print("hash('banana') =", hash('banana'))
print("hash(42)       =", hash(42))
print("hash(3.14)     =", hash(3.14))
print("hash((1, 2))   =", hash((1, 2)))
print()

# Same input ALWAYS gives the same hash (within a session)
print("Consistent?", hash('apple') == hash('apple'))  # True

**Figure 1.2** — Not everything is hashable

In [ ]:
# Mutable objects like lists and dicts are NOT hashable
try:
    hash([1, 2, 3])
except TypeError as e:
    print(f"Error: {e}")

try:
    hash({'a': 1})
except TypeError as e:
    print(f"Error: {e}")

# Why? Because if the object changes, its hash would change,
# and we'd lose track of where we stored it!

### Key Properties of a Good Hash Function

| Property | Meaning |
|----------|--------|
| **Deterministic** | Same input always produces the same output |
| **Fast** | Computing the hash should be quick |
| **Uniform distribution** | Outputs should spread evenly across the range |
| **Minimise collisions** | Different inputs should (ideally) give different outputs |

---
## Part 2: Hash Function Concept — Input → Number → Index

A hash function does two things:

1. **Convert the key to a number** (the *hash value*)
2. **Map that number to a valid index** using modulo (`%`)

```
key  ───▶  hash(key)  ───▶  hash_value % table_size  ───▶  index
```

**Figure 2.1** — Mapping keys to table indices with modulo

In [ ]:
TABLE_SIZE = 10  # Our table has 10 slots (indices 0-9)

keys = ["apple", "banana", "cherry", "date", "elderberry"]

for key in keys:
    h = hash(key)
    index = h % TABLE_SIZE
    print(f"'{key}' -> hash={h} -> index={index}")

**Figure 2.2** — A simple custom hash function for strings

In [ ]:
def simple_hash(key, table_size):
    """Sum the ASCII values of all characters, then mod by table_size."""
    total = 0
    for char in key:
        total += ord(char)
    return total % table_size

TABLE_SIZE = 10
for fruit in ["apple", "banana", "cherry", "date", "fig"]:
    idx = simple_hash(fruit, TABLE_SIZE)
    print(f"simple_hash('{fruit}', {TABLE_SIZE}) = {idx}")

---
### ⏱️ Checkpoint 1 of 3 — Think · Pair · Explain

For **Week 7: Hashing — Intuition & Indexing — PHASE 3 "Choosing Between O(n), O(log n), O(1)"**, state the key invariant, operation cost, or decision rule in your own words.

First write a private prediction. Then explain it to a partner, revise it, and enter your final explanation below. Ask a question now if any step is unclear.


In [ ]:
checkpoint_1_response = ""  # write at least 8 words
studio_pulse(1, checkpoint_1_response)

---
## Part 3: How Python `dict` Works Under the Hood (Simplified)

A Python dictionary is essentially a **hash table**:

1. When you do `d[key] = value`, Python computes `hash(key)`, finds an index, and stores `(key, value)` there.
2. When you do `d[key]`, Python computes `hash(key)`, jumps to that index, and returns the value.

This is why `dict` lookup is **O(1) on average** — no matter how many items, it jumps straight to the right slot.

| Operation | `list` (search) | `dict` (hash table) |
|-----------|-----------------|---------------------|
| Find an item | O(n) | O(1) average |
| Insert | O(1) at end | O(1) average |
| Delete | O(n) | O(1) average |

**Figure 3.1** — Observing O(1) dict lookup vs O(n) list search

In [ ]:
import time

# Create a large list and a dict with the same data
n = 1_000_000
big_list = list(range(n))
big_dict = {i: True for i in range(n)}

# Search for the LAST element (worst case for list)
target = n - 1

# Time list search
start = time.perf_counter()
_ = target in big_list
list_time = time.perf_counter() - start

# Time dict lookup
start = time.perf_counter()
_ = target in big_dict
dict_time = time.perf_counter() - start

print(f"List search: {list_time:.6f} seconds")
print(f"Dict lookup: {dict_time:.6f} seconds")
print(f"Dict is ~{list_time/dict_time:.0f}x faster!")

---
## Part 4: Collisions — When Two Keys Hash to the Same Spot

A **collision** happens when two different keys produce the same index.

Think of it like two students assigned the same locker — we need a strategy!

### Common Collision Resolution Strategies

| Strategy | How it works |
|----------|-------------|
| **Chaining** | Each slot holds a *list* of items. Colliding items go in the same list. |
| **Open addressing** | If a slot is taken, probe the next slot (linear probing). |

**Figure 4.1** — Demonstrating a collision with our simple hash

In [ ]:
# These two strings might collide with our simple hash
TABLE_SIZE = 10

words = ["abc", "bca", "cab", "bac"]  # same letters, different order
for w in words:
    idx = simple_hash(w, TABLE_SIZE)
    print(f"simple_hash('{w}') = {idx}")

# They all have the same sum of ASCII values -> COLLISION!

**Figure 4.2** — Visualising chaining: each slot is a list

In [ ]:
# Simple chaining demonstration
TABLE_SIZE = 7
table = [[] for _ in range(TABLE_SIZE)]  # 7 empty buckets

data = [("apple", 3.5), ("banana", 1.2), ("cherry", 4.0),
        ("date", 5.5), ("fig", 2.0), ("grape", 3.0)]

for key, value in data:
    idx = simple_hash(key, TABLE_SIZE)
    table[idx].append((key, value))
    print(f"'{key}' -> slot {idx}")

print("\n--- Hash Table Contents ---")
for i, bucket in enumerate(table):
    print(f"  Slot {i}: {bucket}")

---
## Part 5: Building a Simple Hash Table from Scratch

Let's build a `SimpleHashTable` class that supports:
- `put(key, value)` — insert or update
- `get(key)` — retrieve a value
- `remove(key)` — delete a key
- `__str__` — pretty print the table

**Figure 5.1** — Complete SimpleHashTable class with chaining

In [ ]:
class SimpleHashTable:
    """A basic hash table using chaining for collision resolution."""

    def __init__(self, size=10):
        self.size = size
        self.table = [[] for _ in range(size)]  # list of buckets
        self.count = 0

    def _hash(self, key):
        """Compute index from key."""
        return hash(key) % self.size

    def put(self, key, value):
        """Insert or update a key-value pair."""
        idx = self._hash(key)
        # Check if key already exists in bucket
        for i, (k, v) in enumerate(self.table[idx]):
            if k == key:
                self.table[idx][i] = (key, value)  # update
                return
        # Key not found, add new pair
        self.table[idx].append((key, value))
        self.count += 1

    def get(self, key):
        """Retrieve value by key. Raises KeyError if not found."""
        idx = self._hash(key)
        for k, v in self.table[idx]:
            if k == key:
                return v
        raise KeyError(f"Key '{key}' not found")

    def remove(self, key):
        """Remove a key-value pair."""
        idx = self._hash(key)
        for i, (k, v) in enumerate(self.table[idx]):
            if k == key:
                self.table[idx].pop(i)
                self.count -= 1
                return v
        raise KeyError(f"Key '{key}' not found")

    def __len__(self):
        return self.count

    def __str__(self):
        lines = []
        for i, bucket in enumerate(self.table):
            if bucket:
                lines.append(f"  [{i}]: {bucket}")
            else:
                lines.append(f"  [{i}]: (empty)")
        return "HashTable(\n" + "\n".join(lines) + "\n)"

**Figure 5.2** — Using our SimpleHashTable

In [ ]:
ht = SimpleHashTable(size=7)

# Insert some student grades
ht.put("Alice", 92)
ht.put("Bob", 85)
ht.put("Charlie", 78)
ht.put("Diana", 95)
ht.put("Eve", 88)

print(ht)
print(f"\nNumber of items: {len(ht)}")
print(f"Alice's grade: {ht.get('Alice')}")

# Update a value
ht.put("Bob", 90)  # Bob improved!
print(f"Bob's new grade: {ht.get('Bob')}")

# Remove a key
ht.remove("Charlie")
print(f"\nAfter removing Charlie: {len(ht)} items")

---
## Part 6: Benchmark — Dict Lookup vs List Search

Let's measure and **plot** how `dict` and `list` perform as data size grows.

> **🔮 Predict first, then measure. Does reality match your prediction?** Before running: predict what the graph will look like. Will dict time stay flat? How fast will list time grow?

**Figure 6.1** — Benchmark: searching for an item in list vs dict at increasing sizes

In [ ]:
import time
import random

sizes = [1000, 5000, 10000, 50000, 100000, 500000, 1000000]
list_times = []
dict_times = []

for n in sizes:
    data_list = list(range(n))
    data_dict = {i: True for i in range(n)}
    target = n - 1  # worst case for list

    # Time list search (average of 3 runs)
    t = 0
    for _ in range(3):
        start = time.perf_counter()
        _ = target in data_list
        t += time.perf_counter() - start
    list_times.append(t / 3)

    # Time dict lookup (average of 3 runs)
    t = 0
    for _ in range(3):
        start = time.perf_counter()
        _ = target in data_dict
        t += time.perf_counter() - start
    dict_times.append(t / 3)

    print(f"n={n:>10,}  list={list_times[-1]:.6f}s  dict={dict_times[-1]:.8f}s")

**Figure 6.2** — Plotting list vs dict lookup times

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.plot(sizes, list_times, 'ro-', label='List search (O(n))', linewidth=2)
plt.plot(sizes, dict_times, 'bs-', label='Dict lookup (O(1))', linewidth=2)
plt.xlabel('Number of Elements (n)', fontsize=12)
plt.ylabel('Time (seconds)', fontsize=12)
plt.title('List Search vs Dict Lookup', fontsize=14)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nObservation: List search time grows linearly with n,")
print("while dict lookup stays nearly constant regardless of size.")

---
### ⏱️ Checkpoint 2 of 3 — Think · Pair · Explain

Predict what happens when the input size doubles. Justify the trend with an operation count or complexity class—not timing alone.

First write a private prediction. Then explain it to a partner, revise it, and enter your final explanation below. Ask a question now if any step is unclear.


In [ ]:
checkpoint_2_response = ""  # write at least 8 words
studio_pulse(2, checkpoint_2_response)

---
## Part 7: Common Errors & Pitfalls

**Figure 7.1** — Unhashable type error

In [ ]:
# ERROR 1: Using a list as a dictionary key
try:
    d = {}
    d[[1, 2, 3]] = "hello"  # lists are mutable -> unhashable
except TypeError as e:
    print(f"Error: {e}")
    print("Fix: Use a tuple instead -> d[(1, 2, 3)] = 'hello'")

# The fix works because tuples are immutable
d = {}
d[(1, 2, 3)] = "hello"
print(f"\nUsing tuple as key works: {d}")

**Figure 7.2** — KeyError when accessing missing keys

In [ ]:
# ERROR 2: Accessing a key that doesn't exist
student_grades = {"Alice": 92, "Bob": 85}

try:
    grade = student_grades["Charlie"]
except KeyError as e:
    print(f"KeyError: {e}")
    print("Fix: Use .get() with a default value")

# Safe way: use .get()
grade = student_grades.get("Charlie", "Not enrolled")
print(f"\nSafe access: Charlie -> {grade}")

**Figure 7.3** — Modifying dict during iteration

In [ ]:
# ERROR 3: Modifying a dict while iterating over it
scores = {"Alice": 50, "Bob": 85, "Charlie": 40, "Diana": 92}

try:
    for name, score in scores.items():
        if score < 60:
            del scores[name]  # Can't modify during iteration!
except RuntimeError as e:
    print(f"RuntimeError: {e}")

# Fix: iterate over a copy of the keys
scores = {"Alice": 50, "Bob": 85, "Charlie": 40, "Diana": 92}
for name in list(scores.keys()):  # list() creates a copy
    if scores[name] < 60:
        del scores[name]
print(f"\nAfter removing low scores: {scores}")

---
## 🌉 Bridge to Next Week

This week we saw how **hashing** gives us O(1) lookups by converting keys to indices. Dictionaries are one of the most powerful data structures in Python.

**Next week**, we'll explore another essential data structure: the **Heap** and **Priority Queue**. While a dict lets you find things instantly, a heap lets you always know the **minimum (or maximum)** element efficiently. This is critical for:

- Scheduling tasks by priority
- Finding the top-k elements
- Algorithms like Dijkstra's shortest path

See you in Week 8!

---
## 🎢 Exercises

Complete the exercises below. Make sure to **run each cell** after writing your solution.

> **🔮 Predict first, then measure. Does reality match your prediction?** For every exercise that involves timing or complexity analysis, make your prediction BEFORE running the code.

### Easy Exercises

**EX1 (Easy):** Use Python's `hash()` function to compute the hash of the strings `"hello"`, `"world"`, and `"python"`. Print each result.

Expected Output (hash values will vary):
```
hash('hello')  = <some integer>
hash('world')  = <some integer>
hash('python') = <some integer>
```

<details><summary>💡 Hint</summary>
Use <code>print(f"hash('hello') = {hash('hello')}")</code> for formatted output.
</details>

In [ ]:
# ✏️ [EX1] Your code here


**EX2 (Easy):** Create a dictionary called `phonebook` with at least 5 name-phone number pairs. Then look up a name that exists and one that doesn't (using `.get()` with a default).

Expected Output:
```
Alice's number: 555-1234
Unknown's number: Not found
```

<details><summary>💡 Hint</summary>
Use <code>phonebook.get("Unknown", "Not found")</code> to safely access a missing key.
</details>

In [ ]:
# ✏️ [EX2] Your code here


**EX3 (Easy):** Write a function `is_hashable(obj)` that returns `True` if the object can be hashed, `False` otherwise. Test it with: `42`, `"hello"`, `[1,2]`, `(1,2)`, `{"a":1}`.

Expected Output:
```
42       -> True
'hello'  -> True
[1, 2]   -> False
(1, 2)   -> True
{'a': 1} -> False
```

<details><summary>💡 Hint</summary>
Use a <code>try/except TypeError</code> block around <code>hash(obj)</code>.
</details>

In [ ]:
# ✏️ [EX3] Your code here


**EX4 (Easy):** Given the list `words = ["cat", "dog", "cat", "bird", "dog", "cat", "fish"]`, use a dictionary to count how many times each word appears. Print the counts.

Expected Output:
```
cat: 3
dog: 2
bird: 1
fish: 1
```

<details><summary>💡 Hint</summary>
Use <code>counts[word] = counts.get(word, 0) + 1</code> to increment counts.
</details>

In [ ]:
# ✏️ [EX4] Your code here


### Medium Exercises

**EX5 (Medium):** Write a better hash function `weighted_hash(key, table_size)` that multiplies each character's ASCII value by its position (index + 1) before summing. Test it on `"abc"`, `"bca"`, `"cab"` with `table_size=10` and show it produces fewer collisions than `simple_hash`.

Expected Output:
```
simple_hash:   abc->X, bca->X, cab->X  (all same!)
weighted_hash: abc->A, bca->B, cab->C  (different!)
```

<details><summary>💡 Hint</summary>
Use <code>enumerate(key)</code> to get both the index and character: <code>total += (i + 1) * ord(char)</code>.
</details>

In [ ]:
# ✏️ [EX5] Your code here


**EX6 (Medium):** Add a `contains(key)` method and a `keys()` method to the `SimpleHashTable` class. `contains` should return `True`/`False`, and `keys` should return a list of all keys.

Expected Output:
```
Contains 'Alice': True
Contains 'Zara': False
All keys: ['Alice', 'Bob', 'Charlie']
```

<details><summary>💡 Hint</summary>
For <code>contains</code>, use a <code>try/except KeyError</code> around <code>self.get(key)</code>. For <code>keys</code>, loop through all buckets and collect the first element of each tuple.
</details>

In [ ]:
# ✏️ [EX6] Your code here


---
### ⏱️ Checkpoint 3 of 3 — Think · Pair · Explain

Choose one completed core exercise. Explain why the algorithm is correct, its dominant cost, and one mechatronics situation where that cost matters.

First write a private prediction. Then explain it to a partner, revise it, and enter your final explanation below. Ask a question now if any step is unclear.


In [ ]:
checkpoint_3_response = ""  # write at least 8 words
studio_pulse(3, checkpoint_3_response)

---
## 🌟 Optional Extension

Exercises 7 and above are enrichment for remaining class time or independent curiosity. They are not homework and are not collected.


**EX7 (Medium):** Write a function `find_duplicates(lst)` that uses a dictionary (or set) to find all duplicate elements in a list. Return the duplicates as a list.

Example: `find_duplicates([1, 3, 5, 3, 7, 1, 9, 5])` should return `[1, 3, 5]` (or any order).

Expected Output:
```
Duplicates: [1, 3, 5]
```

<details><summary>💡 Hint</summary>
Use a <code>seen</code> set. For each item, if it's already in <code>seen</code>, add it to a <code>duplicates</code> set. Otherwise, add it to <code>seen</code>.
</details>

In [ ]:
# ✏️ [EX7] Your code here


**EX8 (Medium):** Write a function `two_sum(nums, target)` that finds two numbers in the list that add up to `target`. Return their indices. Use a dictionary for O(n) solution.

Example: `two_sum([2, 7, 11, 15], 9)` should return `(0, 1)` because `nums[0] + nums[1] = 2 + 7 = 9`.

Expected Output:
```
Indices: (0, 1)
```

<details><summary>💡 Hint</summary>
For each number, compute <code>complement = target - num</code>. Check if <code>complement</code> is already in a dictionary that maps values to indices.
</details>

In [ ]:
# ✏️ [EX8] Your code here


**EX9 (Medium):** Write a function `group_anagrams(words)` that groups words that are anagrams of each other. Use a dictionary where the key is the sorted letters.

Example: `group_anagrams(["eat", "tea", "tan", "ate", "nat", "bat"])` should return groups like `[['eat', 'tea', 'ate'], ['tan', 'nat'], ['bat']]`.

Expected Output:
```
Groups: [['eat', 'tea', 'ate'], ['tan', 'nat'], ['bat']]
```

<details><summary>💡 Hint</summary>
Sort each word's characters to create a key: <code>key = tuple(sorted(word))</code>. Use this as the dictionary key and append words to the corresponding list.
</details>

In [ ]:
# ✏️ [EX9] Your code here


**EX10 (Medium):** Measure and compare the time to check membership (`x in collection`) for a `list` and a `set` with 100,000 elements. Search for 1,000 random elements and print the total time for each.

Expected Output:
```
List: X.XXXX seconds
Set:  X.XXXX seconds
Set is ~Xx faster
```

<details><summary>💡 Hint</summary>
Create a list and set from <code>range(100000)</code>. Use <code>random.randint(0, 100000)</code> to generate search targets. Time a loop of 1000 lookups for each.
</details>

In [ ]:
# ✏️ [EX10] Your code here


### Challenge Exercises

**EX11 (Challenge):** Build a `HashTableLP` class that uses **linear probing** (open addressing) instead of chaining. When a collision occurs, check the next slot, then the next, and so on. Implement `put`, `get`, and `remove`.

Test it by inserting 5 items into a table of size 7 and printing the internal table.

<details><summary>💡 Hint</summary>
Use a fixed-size list of <code>None</code> values. On collision, increment the index with <code>(idx + 1) % size</code> until you find an empty slot. For <code>remove</code>, you can use a sentinel value like <code>"DELETED"</code> to mark removed slots.
</details>

In [ ]:
# ✏️ [EX11] Your code here


**EX12 (Challenge):** Write a function `load_factor_experiment()` that creates hash tables (using `SimpleHashTable`) with increasing numbers of items relative to table size. For each load factor (0.1, 0.3, 0.5, 0.7, 0.9), measure the average number of comparisons needed to find an item. Plot load factor vs average comparisons.

<details><summary>💡 Hint</summary>
Modify the <code>get</code> method to count comparisons. Use a table of size 100 and insert 10, 30, 50, 70, 90 items respectively. For each load factor, search for all inserted keys and compute the average comparisons.
</details>

In [ ]:
# ✏️ [EX12] Your code here
